# Microproyecto 2: Identificación de Relaciones Semánticas con PLN y ML

---

## A. Objetivo
Desarrollar una solución basada en técnicas de **procesamiento de lenguaje natural (PLN)** y **machine learning (ML)** que facilite la interpretación y análisis de información textual para la identificación de relaciones semánticas con los **Objetivos de Desarrollo Sostenible (ODS)**.

---

## B. Conjunto de Datos
El conjunto de datos forma parte del proyecto [OSDG Community Dataset (OSDG-CD)](https://osdg.ai/news/New-release-of-OSDG-Community-dataset) en su versión 2023, que contiene un total de **40.067 textos**, de los cuales **3.000** provienen de fuentes relacionadas con las Naciones Unidas. También contiene documentos públicos, resúmenes de artículos y reportes.

La plataforma reúne investigadores, expertos en la materia y defensores de los ODS de todo el mundo para crear una fuente amplia y precisa de información textual sobre los ODS. Los voluntarios de la comunidad utilizan la plataforma para participar en ejercicios de etiquetado en los que validan la relevancia de cada texto para los ODS basándose en sus conocimientos previos.

> **Traducción y Aumentación de Datos:**
> - Los textos utilizados en este proyecto han sido traducidos al español mediante herramientas como [DeepL](https://www.deepl.com/es/translator).
> - Se realizó aumentación de textos a través de la API de [ChatGPT / OpenAI](https://chat.openai.com/g/g-I1XNbsyDK-api-docs).

---

## C. Actividades a Realizar

1. **Preparación de los textos:**
   - Utilizar el esquema de bolsa de palabras (**BoW**) con pesado **TF-IDF**.
   - Construir un **pipeline** que integre todas las transformaciones y preprocesamientos que se consideren adecuados.

2. **Modelado de Tópicos (LSA):**
   - A partir de la matriz TF-IDF construida, aplicar el algoritmo SVD truncado (`TruncatedSVD` de scikit-learn) para obtener un modelo de tópicos mediante **Análisis Semántico Latente (LSA)**.
   - Explorar un número reducido de componentes (por ejemplo, entre 10 y 20).
   - Para al menos **5 componentes**, identificar y mostrar las palabras con mayor peso a modo de *tópicos*.
   - Interpretar cualitativamente si estos tópicos guardan relación con algunos de los 17 ODS trabajados en el proyecto.

3. **Desarrollo del Modelo de Clasificación:**
   - Construir un modelo de clasificación que permita relacionar un texto con su respectivo ODS.
   - Para manejar la complejidad del espacio de entrada, se puede reutilizar la descomposición SVD (LSA) o aplicar otra técnica de reducción de dimensionalidad pertinente.

4. **Evaluación del Modelo:**
   - Evaluar el modelo con un conjunto de prueba (textos no utilizados durante la etapa de entrenamiento/aprendizaje).

---

## D. Consideraciones
El algoritmo de clasificación a utilizar, así como la técnica de reducción de la dimensionalidad, queda a consideración de cada grupo; sin embargo, **es fundamental justificar la elección de cada técnica**.

---

## E. Entregable
- **Archivos:** Notebook en formatos `.ipynb` y `.html` con el método desarrollado.
- **Documentación:** El notebook debe estar completamente documentado con las justificaciones de las decisiones tomadas en cada paso.
- **Ejecución visible:** Deben ser visibles las salidas y ejecuciones de cada celda.
- **Evidencia práctica:** Para evidenciar el desempeño del método construido, el notebook debe mostrar las clasificaciones para al menos **4 textos del conjunto de test**.
- **Plazo:** Entrega al final de la **Semana 7** en el espacio correspondiente.

---

## F. Criterios de Evaluación

| Actividad | Porcentaje |
| :--- | :---: |
| **Preparación de los datos**, incluyendo la reducción de la dimensionalidad y justificación de decisiones tomadas. | **30%** |
| **Construcción del pipeline** de preparación de datos. | **15%** |
| **Construcción del modelo de clasificación** con el algoritmo seleccionado, búsqueda de hiperparámetros y validación con medidas de evaluación adecuadas (justificando algoritmo, métricas y reducción de dimensionalidad). | **30%** |
| **Evidencia del desempeño** del modelo mostrando clasificaciones sobre un conjunto de textos no utilizados durante el aprendizaje. | **10%** |
| **Construcción del modelo LSA** sobre la matriz TF-IDF e interpretación cualitativa de al menos 5 tópicos frente a los ODS. | **15%** |
| **Total** | **100%** |

---

## G. Bonificación Adicional (Opcional — +15 Puntos)
Con el propósito de fortalecer las competencias en despliegue y aplicación práctica de modelos de ML, se otorgará una bonificación adicional de **15 puntos** a los grupos que implementen el modelo en una aplicación interactiva utilizando **Streamlit**.

### Requisitos de la Aplicación:
- Permitir al usuario ingresar un texto libre.
- Procesar el texto utilizando el mismo pipeline construido en el proyecto.
- Generar como salida la predicción del Objetivo de Desarrollo Sostenible (ODS) correspondiente.
- Ser totalmente funcional y ejecutarse correctamente.

> *Nota:* La bonificación es voluntaria y no reemplaza los criterios de la rúbrica; premia el paso del entorno experimental al despliegue práctico.


# Importe Librerias

In [1]:
import warnings
import pandas as pd
from stop_words import get_stop_words
import re as regexExpression
import string
from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')

In [2]:
#Funciones comunes
def limpiar_texto(textos):
    """
    Limpia y normaliza texto en español:
    - Minúsculas
    - Remoción de caracteres invisibles / zero-width (\u200b, \ufeff, etc.)
    - Remoción de URLs y etiquetas HTML
    - Remoción de signos de puntuación y dígitos
    - Normalización de espacios en blanco
    """
    if not isinstance(textos, str):
        return ""
    
    # Convertir a minúsculas
    textos = textos.lower()
    
    # Eliminar caracteres invisibles / zero-width (incluyendo \u200b)
    textos = regexExpression.sub(r'[\u200b\ufeff\u200e\u200f\u202a-\u202e]', '', textos)
    
    # Eliminar URLs
    textos = regexExpression.sub(r'https?://\S+|www\.\S+', '', textos)
    
    # Eliminar etiquetas HTML
    textos = regexExpression.sub(r'<.*?>', '', textos)
    
    # Eliminar números y caracteres de puntuación especiales
    textos = regexExpression.sub(r'\d+', '', textos)
    textos = textos.translate(str.maketrans('', '', string.punctuation + '«»“”¿¡'))
    
    # Eliminar espacios múltiples
    textos = regexExpression.sub(r'\s+', ' ', textos).strip()
    
    return textos


def stemmed_tokenizer(text):
    stemmer_es = SnowballStemmer('spanish')
    return [stemmer_es.stem(word) for word in text.split()]



# Carga de datos

### Análisis de Resultados y Conclusiones

#### 1. Comparación del Modelo Básico vs. Optimizado
Al realizar la búsqueda de hiperparámetros con `RandomizedSearchCV`, el modelo optimizado logró una mejora general en el rendimiento frente al modelo base con parámetros por defecto:
* **Accuracy:** Aumentó de 80.99% a 81.49% (±0.16%).
* **Precision:** Pasó de 63.52% a 65.27%, disminuyendo los falsos positivos.
* **Recall y F1-Score:** El Recall subió a 39.18% y el F1-Score alcanzó 48.96% (frente a 47.48% del modelo base).
* **Tiempo de ejecución:** Ambos modelos entrenaron en aproximadamente 0.5 segundos, demostrando que la optimización no incrementó el costo computacional.

#### 2. Impacto de los Hiperparámetros Seleccionados
* **`max_depth = 4`** (por defecto 6): Al limitar la profundidad de los árboles, se restringe la complejidad individual de cada estimador, reduciendo el sobreajuste (*overfitting*) en el conjunto de validación.
* **`learning_rate = 0.035`** (por defecto 0.3): Una tasa de aprendizaje más baja permite que las correcciones en cada iteración de boosting sean más graduales, mejorando la capacidad de generalización.
* **`n_estimators = 166`** (por defecto 100): Compensa la menor tasa de aprendizaje, permitiendo que el ensamble converja adecuadamente.
* **`reg_lambda = 0.11`** (por defecto 1.0): Proporciona regularización L2 sobre las hojas, controlando la magnitud de los pesos y suavizando las predicciones.

#### 3. Impacto de la Ponderación de Costos (`scale_pos_weight`)
Dado el desbalance de clases en el dataset (77.3% negativos vs. 22.7% positivos), se implementó un esquema de costos mediante `scale_pos_weight = 3.40` (ratio negativos/positivos):
* **Efecto en las métricas:** El Recall aumentó significativamente de 39.18% a **67.30%**, logrando que el modelo detecte una proporción mucho mayor de defectos. Como contrapartida esperada (*trade-off*), la Precision disminuyó a 46.90% y el Accuracy a 75.32%, mientras que el F1-Score mejoró a **55.28%**.
* **Justificación en el problema:** En la detección de defectos de software, un falso negativo (no detectar un fallo que pasa a producción) tiene un costo crítico mucho mayor que un falso positivo (revisar un módulo que no tenía defectos). Por ello, el modelo con costos es el más adecuado para este problema de negocio.

In [3]:
data_path = 'data/Datos_textosODS.xlsx'
df = pd.read_excel(data_path)

df.head(5)


,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5


In [4]:
df['texto_limpio'] = df['textos'].apply(limpiar_texto)
df['texto_limpio'].head(5)

0    aprendizaje y educación se consideran sinónimo...
1    no dejar clara la naturaleza de estos riesgos ...
2    como resultado un mayor y mejorado acceso al a...
3    con el congreso firmemente en control de la ju...
4    luego dos secciones finales analizan las impli...
Name: texto_limpio, dtype: object

In [5]:
spanish_stopwords = get_stop_words('spanish')

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=stemmed_tokenizer, #Tokenizador personalizado para aplicar stemming
    stop_words=spanish_stopwords,
    max_features=10000,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_tfidf = tfidf_vectorizer.fit_transform(df['texto_limpio'])

print('Feature names:')
print(tfidf_vectorizer.get_feature_names_out())

print('primeras 10 características:')
print(tfidf_vectorizer.get_feature_names_out()[:10])




Feature names:
['abaj' 'abaj haci' 'abandon' ... 'º' '–' '•']
primeras 10 características:
['abaj' 'abaj haci' 'abandon' 'abarc' 'abastec' 'abastec agu' 'abiert'
 'abog' 'abol' 'abord']
